# Create ML-ready dataset from FH-EARLY export

> Goal: build a reusable pipeline that converts exported eCRF tables into one ML dataframe
> 
> NB: This notebook uses mock data only to test the structure.. final modelling decisions will be made after receiving the real dataset
> 
> We will use: 
> - export `.csv`, which contain the actual data row
> - `master_catalog`, that we created in `tutorial_catalog.ipynb`that tell us how columns should be interpreted: 
>   - identify numerical, categorical, checkbox, date and texts
>   - chose analysis ready columns
>   - keep track of feature meaning

In [17]:
import importlib
from pathlib import Path
import utils_database

importlib.reload(utils_database)

from utils_database import *

export_folder = Path("export-example")

master_catalog = pd.read_pickle("master_catalog.pkl")

### Load catalog

In [9]:
master_catalog.shape
master_catalog.head()

,table,column,sas_label,sas_informat,Référence,Id,Saisie,Type,Format,Intitulé,Type_english,code_labels
0,INC,STUDY_ID,Identification étude,$8.,STUDY_ID,NaN,NaN,NaN,NaN,NaN,unknown,{}
1,INC,COUNTRY_ID,Identifiant pays,$4.,COUNTRY_ID,NaN,NaN,NaN,NaN,NaN,unknown,{}
2,INC,EXTRACTION_DATE,Date extraction,$19.,EXTRACTION_DATE,NaN,NaN,NaN,NaN,NaN,unknown,{}
3,INC,SITE_ID,Centre,$4.,SITE_ID,NaN,NaN,NaN,NaN,NaN,unknown,{}
4,INC,SUBJECT_ID,Identifiant unique,$4.,SUBJECT_ID,NaN,NaN,NaN,NaN,NaN,unknown,{}


In [10]:
master_catalog[["Type_english"]].value_counts(dropna=False)
# If you want to analize the unknown types, you can do it like this:
# master_catalog[master_catalog["Type_english"]== "unknown"]
# Probably these are not unknown because parsing failed, but because they are administrative identifiers, technical export cokumns, module and do not belong to the clinical dictionary. 

Type_english
numeric         124
categorical      87
date             64
checkbox         54
unknown          37
text             15
Name: count, dtype: int64

### Load the `.csv file`

In [11]:
inc = read_export_csv(export_folder / "INC_ANSI.csv")
ml = read_export_csv(export_folder / "ML_ANSI.csv")
tab_bs = read_export_csv(export_folder / "TAB_BS_ANSI.csv")
tab_fh = read_export_csv(export_folder / "TAB_FH_ANSI.csv")

export_tables = {"INC": inc, "ML": ml, "TAB_BS": tab_bs, "TAB_FH": tab_fh}

### Create table-specific analysis datasets
> Before merging everything into one dataset, we create one analysis-ready dataset per exported table, so we can: 
> - inspect each module indipendentely
> - test preprocessing rule 
> - then use identifiers to merge
> 
> To create the first version of the dataset we use `create_table_dataset` and we keep only variable which `Type_english` (information that can be found in the `catalog`) is not unknown AND `_ID` variables to then merge the dataset


In [12]:
master_catalog["table"].value_counts()

table
ML        272
INC        73
TAB_BS     20
TAB_FH     16
Name: count, dtype: int64

In [13]:
inc_dataset = create_table_dataset(inc, "INC", master_catalog)
ml_dataset = create_table_dataset(ml, "ML", master_catalog)
tab_bs_dataset = create_table_dataset(tab_bs, "TAB_BS", master_catalog)
tab_fh_dataset = create_table_dataset(tab_fh, "TAB_FH", master_catalog)

In [14]:
print("INC:", inc_dataset.shape)
print("ML:", ml_dataset.shape)
print("TAB_BS:", tab_bs_dataset.shape)
print("TAB_FH:", tab_fh_dataset.shape)

INC: (5, 68)
ML: (5, 268)
TAB_BS: (5, 14)
TAB_FH: (5, 10)


### Inspect the generated datasets
> The generated datasets contain only columns considered potentially useful for analysis:
> - numeric
> - categorical
> - checkbox
> - date
> - text variables
> 
> Technical/system columns excluded from the metadata catalog are not included

In [15]:
inc_dataset.head()

,STUDY_ID,COUNTRY_ID,SITE_ID,SUBJECT_ID,SUBJECT_REF,SEX,BRTHDAT_D,BRTHDAT_M,BRTHDAT_Y,BRTHDAT,...,CSTTR1AYN,CICZ1YN,CIFR1YN,CIFR2YN,CIFR3YN,CNIFR1YN,CIPT1YN,CIPT2YN,CITR1YN,CITR2YN
0,FH-EARLY,fr,1,c1/1,ZZOI-0001,1.0,NaN,7,1997.0,NaN,...,NaN,NaN,1.0,1.0,1,0,NaN,NaN,NaN,NaN
1,FH-EARLY,fr,2,c1/2,QGYD-0002,1.0,NaN,6,2007.0,NaN,...,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN
2,FH-EARLY,fr,3,c1/3,EJI-0003,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,1.0,1.0,1,0,NaN,NaN,NaN,NaN
3,FH-EARLY,fr,4,c1/4,CH-0004,2.0,NaN,4,1978.0,NaN,...,NaN,NaN,1.0,1.0,1,0,NaN,NaN,NaN,NaN
4,FH-EARLY,fr,5,c1/5,UHVT-0005,2.0,NaN,ND,1972.0,NaN,...,NaN,NaN,0.0,1.0,1,0,NaN,NaN,NaN,NaN


In [19]:
ml_dataset.head()

,STUDY_ID,COUNTRY_ID,SITE_ID,SUBJECT_ID,SUBJECT_REF,CD1TXT,CD2LS,CD3NUM,CD3NUM_V,CD4NUM,...,TTT4LS_C1,TTT4LS_C2,TTT4LS_C3,TTT4LS_C4,TTT4LS_C5,TTT4LS_C6,TTT4LS_C7,VS1YN,VS1ALS,VS1BTXT
0,FH-EARLY,fr,1,c1/1,ZZOI-0001,SIOIQBDS,2,66,66.0,150.823,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN
1,FH-EARLY,fr,2,c1/2,QGYD-0002,BGRXGOEYMZRG,2,113,113.0,111.952,...,1.0,1.0,NaN,NaN,NaN,NaN,NaN,1,3.0,NaN
2,FH-EARLY,fr,3,c1/3,EJI-0003,NXQBDZBPXB,2,106,106.0,133.242,...,1.0,1.0,1.0,1.0,1.0,NaN,NaN,1,6.0,NaN
3,FH-EARLY,fr,4,c1/4,CH-0004,XWRXMDZTWJ,1,74,74.0,196.870,...,1.0,1.0,1.0,NaN,NaN,NaN,NaN,0,NaN,NaN
4,FH-EARLY,fr,5,c1/5,UHVT-0005,RO,2,123,123.0,NaN,...,1.0,1.0,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN


## Handle variable representations

> Different exported variable types require different preprocessing strategies before creating ml datasets
>
> Examples:
> - date variables may need conversion into timestamps or derived temporal features
> - categorical variables may require label or one-hot encoding
> - checkbox variables are already expanded into binary columns
> - numeric variables may require selecting the `_V` analysis-ready columns
> 
> Function: 
> - `get_numeric_cols(master_catalog, table_name)` to get numerical columns
> -  `get_categorical_cols(master_catalog, table_name)` to get categorical columns

In [5]:
ml_numeric_cols = get_numeric_cols(master_catalog, "ML")

In [6]:
ml_categorical_cols = get_categorical_cols(master_catalog, "ML")

In [10]:
ml_checkbox_cols = get_checkbox_cols(master_catalog, "ML")

In [15]:
inc_date_cols = get_date_cols(master_catalog, "INC")

In [18]:
ml_text_cols = get_text_cols(master_catalog, "ML")

In [19]:
ml_text_cols

['SUBJECT_REF', 'CD1TXT', 'BS1A1TXT', 'BS34A1TXT', 'BS35A1TXT', 'VS1BTXT']